# Which map is random? Local-attraction replication

This notebook reproduces the 20 mirrored stimuli and the exact model prompt used in the article. The generator is deterministic. No API request runs unless `RUN_MODEL` is set explicitly.

In [ ]:
import base64, io, os, random, re, requests
import matplotlib.pyplot as plt
import pandas as pd

N = 120
ATTRACTION = 0.05
RADIUS = 0.07
STEPS = 12
MASTER_SEED = 7
BASE_TRIALS = 10

QUESTION = (
    'The image shows two squares of dots, labelled LEFT and RIGHT. '
    'Exactly one of them was generated by a truly random process; the other was '
    'generated by a different process. Which square was generated by the truly '
    'random process?\n\nAnswer with exactly one word on the first line: LEFT or '
    'RIGHT. Then one short sentence saying what made you choose it.'
)

In [ ]:
# Exact browser figure: Python port of the component's Mulberry32 generator.
def mulberry32(seed):
    state = seed & 0xffffffff
    def draw():
        nonlocal state
        state = (state + 0x6D2B79F5) & 0xffffffff
        t = ((state ^ (state >> 15)) * (1 | state)) & 0xffffffff
        t = (((t + (((t ^ (t >> 7)) * (61 | t)) & 0xffffffff)) & 0xffffffff) ^ t) & 0xffffffff
        return ((t ^ (t >> 14)) & 0xffffffff) / 4294967296
    return draw

def browser_uniform(seed, n=N):
    rng = mulberry32(seed)
    return [(rng(), rng()) for _ in range(n)]

def article_pair(seed=20260826):
    side_rng = mulberry32(seed)
    random_index = 0 if side_rng() < 0.5 else 1
    uniform = browser_uniform(seed ^ 0x9E3779B9)
    clustered = attract(browser_uniform(seed ^ 0x85EBCA6B))
    return ((uniform, clustered) if random_index == 0 else (clustered, uniform), random_index)

In [ ]:
def uniform_points(rng, n=N):
    return [(rng.random(), rng.random()) for _ in range(n)]

def torus_delta(a, b):
    d = b - a
    if d > 0.5: d -= 1.0
    if d < -0.5: d += 1.0
    return d

def attract(start, strength=ATTRACTION, radius=RADIUS, steps=STEPS):
    points = list(start)
    radius2 = radius * radius
    for _ in range(steps):
        moves = [[0.0, 0.0, 0.0] for _ in points]
        for i in range(len(points)):
            for j in range(i + 1, len(points)):
                dx = torus_delta(points[i][0], points[j][0])
                dy = torus_delta(points[i][1], points[j][1])
                d2 = dx*dx + dy*dy
                if d2 == 0.0 or d2 >= radius2: continue
                weight = 1.0 - d2**0.5 / radius
                moves[i][0] += dx * weight; moves[i][1] += dy * weight; moves[i][2] += weight
                moves[j][0] -= dx * weight; moves[j][1] -= dy * weight; moves[j][2] += weight
        updated = []
        for point, move in zip(points, moves):
            if move[2] == 0.0:
                updated.append(point)
            else:
                updated.append(((point[0] + strength*move[0]/move[2]) % 1.0,
                                (point[1] + strength*move[1]/move[2]) % 1.0))
        points = updated
    return points

def render_pair(left, right):
    fig, axes = plt.subplots(1, 2, figsize=(10, 5.2), dpi=110)
    for ax, points, label in zip(axes, (left, right), ('LEFT', 'RIGHT')):
        ax.scatter([p[0] for p in points], [p[1] for p in points], s=14, c='#1e293b')
        ax.set(xlim=(0, 1), ylim=(0, 1), xticks=[], yticks=[])
        ax.set_title(label, fontsize=13, color='#334155')
        for spine in ax.spines.values(): spine.set_color('#cbd5e1')
    fig.tight_layout()
    return fig

def png_bytes(left, right):
    fig = render_pair(left, right)
    out = io.BytesIO()
    fig.savefig(out, format='png', facecolor='white', metadata={'Software': 'matplotlib'})
    plt.close(fig)
    return out.getvalue()

(article_left, article_right), article_random_index = article_pair()
display(render_pair(article_left, article_right))
print('The article random panel is', ('LEFT', 'RIGHT')[article_random_index])

In [ ]:
master = random.Random(MASTER_SEED)
episodes = []
for index in range(BASE_TRIALS):
    seed = master.randrange(10**9)
    uniform = uniform_points(random.Random(seed))
    clustered = attract(uniform_points(random.Random(seed ^ 0x5DEECE66D)))
    uniform_left = master.random() < 0.5
    left, right = (uniform, clustered) if uniform_left else (clustered, uniform)
    answer = 'LEFT' if uniform_left else 'RIGHT'
    episodes.append({'id': f't{index:02d}', 'left': left, 'right': right, 'answer': answer})
    episodes.append({'id': f't{index:02d}m', 'left': right, 'right': left,
                     'answer': 'RIGHT' if answer == 'LEFT' else 'LEFT'})

display(render_pair(episodes[0]['left'], episodes[0]['right']))
print(QUESTION)
print(f'{len(episodes)} episodes; answer to first episode: {episodes[0]["answer"]}')

## Optional paid rerun

Set an OpenRouter key in Colab Secrets as `OPENROUTER_API_KEY`, set `RUN_MODEL` to one model ID, and execute the cell. This makes 20 paid requests. The default is `None`.

In [ ]:
RUN_MODEL = None  # e.g. 'google/gemini-3.7-flash'

def parse_choice(text):
    first = (text or '').strip().upper().split('\n')[0]
    match = re.search(r'\b(LEFT|RIGHT)\b', first)
    return match.group(1) if match else None

def ask_openrouter(model, episode, api_key):
    encoded = base64.b64encode(png_bytes(episode['left'], episode['right'])).decode()
    body = {
        'model': model,
        'messages': [{'role': 'user', 'content': [
            {'type': 'text', 'text': QUESTION},
            {'type': 'image_url', 'image_url': {'url': 'data:image/png;base64,' + encoded}},
        ]}],
        'reasoning': {'max_tokens': 1024},
        'max_tokens': 1600,
        'usage': {'include': True},
    }
    response = requests.post('https://openrouter.ai/api/v1/chat/completions', json=body,
                             headers={'Authorization': f'Bearer {api_key}'}, timeout=180)
    response.raise_for_status()
    payload = response.json()
    text = payload['choices'][0]['message'].get('content') or ''
    return text, parse_choice(text), (payload.get('usage') or {}).get('cost', 0)

if RUN_MODEL:
    try:
        from google.colab import userdata
        api_key = userdata.get('OPENROUTER_API_KEY')
    except ImportError:
        api_key = os.environ['OPENROUTER_API_KEY']
    rows = []
    for episode in episodes:
        raw, choice, cost = ask_openrouter(RUN_MODEL, episode, api_key)
        rows.append({'episode': episode['id'], 'truth': episode['answer'], 'choice': choice,
                     'correct': choice == episode['answer'], 'cost_usd': cost, 'raw': raw})
    rerun = pd.DataFrame(rows)
    display(rerun)
    print('score:', int(rerun.correct.sum()), '/20')
    print('reported cost: $', rerun.cost_usd.sum())
else:
    print('No API calls made. Set RUN_MODEL to opt in.')

In [ ]:
published = pd.DataFrame([
    ('google/gemini-3.7-flash', 13, 0.025533375),
    ('anthropic/claude-opus-5', 10, 0.27625),
    ('openai/gpt-5.6-sol', 7, 0.04769),
    ('anthropic/claude-sonnet-5', 4, 0.05257),
    ('moonshotai/kimi-k3', 0, 0.2013924),
    ('qwen/qwen3.8-max', 0, 0.076974),
], columns=['model', 'correct_of_20', 'reported_cost_usd'])
published